Data cleaning and feature transformation pipeline for meteorite landings, fireballs, and asteroid datasets.


In [1]:
import os
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve directory paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATASETS_DIR = PROJECT_ROOT / "datasets"
TRANSFORMED_DIR = DATASETS_DIR / "transformed"
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Output dir   : {TRANSFORMED_DIR}")

Project root : /home/ahmad/projects/spaceapp
Output dir   : /home/ahmad/projects/spaceapp/datasets/transformed


Load raw meteorite records, CNEOS fireballs, and API snapshots.


In [2]:
# 1. Load Raw Meteorite Landings
raw_meteorites_path = DATASETS_DIR / "Meteorite_Landings.csv"
df_raw_meteorites = pd.read_csv(raw_meteorites_path)
print(f"[Load] Meteorite Landings Raw : {len(df_raw_meteorites):,} rows")

# 2. Load CNEOS Fireball Data
raw_fireball_path = DATASETS_DIR / "cneos_fireball_data.csv"
df_raw_fireball = pd.read_csv(raw_fireball_path, header=[0, 1])
print(f"[Load] CNEOS Fireballs Raw    : {len(df_raw_fireball):,} rows")

# 3. Load API Snapshots (JSON)
with open(DATASETS_DIR / "cneos_sentry" / "sentry.json") as f:
    sentry_raw = json.load(f)
with open(DATASETS_DIR / "sbdb_cad" / "cad.json") as f:
    cad_raw = json.load(f)

print(f"[Load] Sentry Tracked Objects : {len(sentry_raw.get('data', [])):,} objects")
print(f"[Load] CAD Close Approaches   : {len(cad_raw.get('data', [])):,} events")

[Load] Meteorite Landings Raw : 45,716 rows
[Load] CNEOS Fireballs Raw    : 1,074 rows
[Load] Sentry Tracked Objects : 2,251 objects
[Load] CAD Close Approaches   : 1,097 events


Clean meteorite landings data by removing null coordinates, Null Island (0,0), and record anomalies.


In [3]:
raw_count = len(df_raw_meteorites)

# 1. Drop Missing Values
df_step1 = df_raw_meteorites.dropna(subset=["reclat", "reclong", "mass (g)"]).copy()
dropped_nan = raw_count - len(df_step1)

# 2. Drop Null Island (0.0, 0.0)
df_step2 = df_step1[~((df_step1["reclat"] == 0.0) & (df_step1["reclong"] == 0.0))].copy()
dropped_null_island = len(df_step1) - len(df_step2)

# 3. Drop Year > 2026 & Mass <= 0
df_clean_meteorites = df_step2[
    (df_step2["year"] <= 2026) & (df_step2["mass (g)"] > 0)
].copy()
dropped_anomalies = len(df_step2) - len(df_clean_meteorites)

final_count = len(df_clean_meteorites)
retention_rate = (final_count / raw_count) * 100

print("=== DATA CLEANING AUDIT TRAIL ===")
print(f"1. Raw Initial Records  : {raw_count:,} rows")
print(f"2. Dropped Missing NaN  : -{dropped_nan:,} rows -> Remaining {len(df_step1):,}")
print(f"3. Dropped Null Island  : -{dropped_null_island:,} rows -> Remaining {len(df_step2):,}")
print(f"4. Dropped Anomalies    : -{dropped_anomalies:,} rows -> Remaining {final_count:,}")
print(f"-> TOTAL CLEAN DATASET  : {final_count:,} rows ({retention_rate:.2f}% retention)")

assert final_count == 31911, f"Expected 31911 rows, got {final_count}"

=== DATA CLEANING AUDIT TRAIL ===
1. Raw Initial Records  : 45,716 rows
2. Dropped Missing NaN  : -7,434 rows -> Remaining 38,282
3. Dropped Null Island  : -6,214 rows -> Remaining 32,068
4. Dropped Anomalies    : -157 rows -> Remaining 31,911
-> TOTAL CLEAN DATASET  : 31,911 rows (69.80% retention)


Map meteorite classifications into 4 physical material composition classes.


In [4]:
def map_composition(rc):
    rc_str = str(rc).strip()
    if any(k in rc_str for k in ["Iron", "Pallasite", "Mesosiderite"]):
        return "Iron / Stony-Iron"
    achondrites = ["Achondrite", "Aubrite", "Diogenite", "Eucrite", "Howardite", "Ureilite", "Angrite", "Brachinite", "Martian", "Lunar"]
    if any(a in rc_str for a in achondrites):
        return "Achondrite"
    if rc_str.startswith("C") and not rc_str.startswith("Chondrite-fusion") and not rc_str.startswith("Chondrite-ung"):
        return "Carbonaceous Chondrite"
    return "Ordinary Chondrite"

df_clean_meteorites["material_class"] = df_clean_meteorites["recclass"].apply(map_composition)

# Physical parameters by material class
PHYSICAL_PARAMS = {
    "Iron / Stony-Iron": {"density_g_cm3": 7.8, "yield_strength_mpa": 350, "regime": "Surface Crater Impact"},
    "Achondrite": {"density_g_cm3": 3.2, "yield_strength_mpa": 65, "regime": "Mid-Atmosphere Breakup (15-25 km)"},
    "Carbonaceous Chondrite": {"density_g_cm3": 2.2, "yield_strength_mpa": 5, "regime": "High-Altitude Airburst (>35 km)"},
    "Ordinary Chondrite": {"density_g_cm3": 3.4, "yield_strength_mpa": 35, "regime": "Tropospheric Shockwave Airburst (15-30 km)"}
}

df_clean_meteorites["density_g_cm3"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["density_g_cm3"])
df_clean_meteorites["yield_strength_mpa"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["yield_strength_mpa"])
df_clean_meteorites["impact_regime"] = df_clean_meteorites["material_class"].map(lambda c: PHYSICAL_PARAMS[c]["regime"])

print("Material Class Distribution:")
print(df_clean_meteorites["material_class"].value_counts())

Material Class Distribution:
material_class
Ordinary Chondrite        28988
Iron / Stony-Iron          1113
Achondrite                  985
Carbonaceous Chondrite      825
Name: count, dtype: int64


Extract logarithmic mass, absolute latitude, and decade grouping features.


In [5]:
df_clean_meteorites["log10_mass_g"] = np.log10(df_clean_meteorites["mass (g)"])
df_clean_meteorites["abs_lat"] = df_clean_meteorites["reclat"].abs()
df_clean_meteorites["decade"] = (df_clean_meteorites["year"] // 10 * 10).astype(int)
df_clean_meteorites["is_fell"] = (df_clean_meteorites["fall"] == "Fell").astype(int)

# Massive Outlier Detection (>35.04 kg / log10_mass > 4.54)
df_clean_meteorites["is_massive_outlier"] = df_clean_meteorites["log10_mass_g"] > 4.5448
print(f"Massive meteorites (>35 kg) : {df_clean_meteorites['is_massive_outlier'].sum():,} objects")

Massive meteorites (>35 kg) : 576 objects


Filter official Indonesian meteorite landing records within national geographical bounds.


In [6]:
df_indonesia = df_clean_meteorites[
    (df_clean_meteorites["reclat"] >= -11.5) & (df_clean_meteorites["reclat"] <= 6.0) &
    (df_clean_meteorites["reclong"] >= 95.0) & (df_clean_meteorites["reclong"] <= 141.0)
].sort_values(by="mass (g)", ascending=False).copy()

# Kinetic energy estimation at v = 17 km/s
def calc_energy(mass_g, velocity_ms=17000):
    mass_kg = mass_g / 1000.0
    joules = 0.5 * mass_kg * (velocity_ms ** 2)
    tons_tnt = joules / 4.184e9
    return round(tons_tnt, 2)

df_indonesia["energy_tons_tnt"] = df_indonesia["mass (g)"].apply(calc_energy)

print(f"Total Indonesian Meteorites: {len(df_indonesia)} records")
for idx, row in df_indonesia.head(5).iterrows():
    print(f"- {row['name']} ({row['year']:.0f}): {row['mass (g)']/1000:.2f} kg, {row['material_class']}, {row['energy_tons_tnt']} Tons TNT")

Total Indonesian Meteorites: 19 records
- Prambanan (1797): 500.00 kg, Iron / Stony-Iron, 17.27 Tons TNT
- Jepara (2008): 499.50 kg, Iron / Stony-Iron, 17.25 Tons TNT
- Djati-Pengilon (1884): 166.00 kg, Ordinary Chondrite, 5.73 Tons TNT
- Jumapalo (1984): 32.49 kg, Ordinary Chondrite, 1.12 Tons TNT
- Meester-Cornelis (1915): 24.75 kg, Ordinary Chondrite, 0.85 Tons TNT


Parse CNEOS fireball cardinal directional coordinates into signed decimal degrees.


In [7]:
def parse_coord(val):
    if pd.isna(val) or not str(val).strip():
        return np.nan
    val_str = str(val).strip()
    m = re.match(r"^([0-9.]+)([NSEW])$", val_str)
    if not m:
        return np.nan
    num = float(m.group(1))
    direction = m.group(2)
    if direction in ["S", "W"]:
        num = -num
    return num

fireball_records = []
for _, row in df_raw_fireball.iterrows():
    dt = row.iloc[0]
    lat = parse_coord(row.iloc[1])
    lon = parse_coord(row.iloc[2])
    alt = float(row.iloc[3]) if pd.notna(row.iloc[3]) and str(row.iloc[3]).strip() else None
    energy_j = float(row.iloc[7]) if pd.notna(row.iloc[7]) and str(row.iloc[7]).strip() else None
    energy_kt = float(row.iloc[8]) if pd.notna(row.iloc[8]) and str(row.iloc[8]).strip() else None
    
    if pd.notna(lat) and pd.notna(lon):
        fireball_records.append({
            "datetime": str(dt),
            "lat": lat,
            "lon": lon,
            "alt_km": alt,
            "energy_j": energy_j,
            "energy_kt": energy_kt
        })

df_clean_fireball = pd.DataFrame(fireball_records)
print(f"CNEOS Fireballs Valid Coordinates: {len(df_clean_fireball)} events")

CNEOS Fireballs Valid Coordinates: 888 events


Normalize Sentry risk and CAD close-approach records into standardized tabular format.


In [8]:
# Sentry Normalization
sentry_list = []
for obj in sentry_raw.get("data", []):
    sentry_list.append({
        "designation": obj.get("des"),
        "fullname": obj.get("fullname"),
        "impact_probability": float(obj.get("ip", 0.0)),
        "diameter_km": float(obj.get("diameter")) if obj.get("diameter") else None,
        "palermo_scale_max": float(obj.get("ps_max")) if obj.get("ps_max") else None,
        "torino_scale_max": int(obj.get("ts_max")) if obj.get("ts_max") else 0,
        "velocity_inf_kms": float(obj.get("v_inf")) if obj.get("v_inf") else None,
        "year_range": obj.get("range")
    })
df_sentry = pd.DataFrame(sentry_list).sort_values(by="impact_probability", ascending=False)
print(f"Normalized Sentry : {len(df_sentry)} objects")

# CAD Normalization
cad_fields = cad_raw.get("fields", [])
cad_rows = cad_raw.get("data", [])
df_cad = pd.DataFrame(cad_rows, columns=cad_fields)
df_cad["dist_ld"] = pd.to_numeric(df_cad["dist"], errors="coerce") * 389.1703
df_cad["dist_km"] = pd.to_numeric(df_cad["dist"], errors="coerce") * 149597870.7
df_cad["v_rel_kms"] = pd.to_numeric(df_cad["v_rel"], errors="coerce")
print(f"Normalized CAD    : {len(df_cad)} approach events")

Normalized Sentry : 2251 objects
Normalized CAD    : 1097 approach events


Export transformed datasets to datasets/transformed for downstream modeling and UI integration.


In [9]:
# 1. Cleaned & Transformed Meteorites (CSV & JSON)
out_csv = TRANSFORMED_DIR / "meteorites_cleaned_transformed.csv"
df_clean_meteorites.to_csv(out_csv, index=False)

out_json = TRANSFORMED_DIR / "meteorites_cleaned_transformed.json"
df_clean_meteorites.to_json(out_json, orient="records", indent=2)

# 2. Indonesia Meteorites
out_indo = TRANSFORMED_DIR / "indonesia_meteorites.json"
df_indonesia.to_json(out_indo, orient="records", indent=2)

# 3. CNEOS Fireballs Cleaned
out_fireball = TRANSFORMED_DIR / "cneos_fireballs_cleaned.json"
df_clean_fireball.to_json(out_fireball, orient="records", indent=2)

# 4. Asteroid API Normalized
out_sentry = TRANSFORMED_DIR / "sentry_risks_transformed.json"
df_sentry.head(100).to_json(out_sentry, orient="records", indent=2)

# 5. CAD Normalized
out_cad = TRANSFORMED_DIR / "cad_approaches_transformed.json"
df_cad.head(100).to_json(out_cad, orient="records", indent=2)

print("=== TRANSFORMED EXPORT SUMMARY ===")
for f in sorted(TRANSFORMED_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"✓ {f.name:38} : {size_kb:8.1f} KB")

=== TRANSFORMED EXPORT SUMMARY ===
✓ cad_approaches_transformed.json        :     41.0 KB
✓ cneos_fireballs_cleaned.json           :    130.6 KB
✓ indonesia_meteorites.json              :     10.2 KB
✓ meteorites_cleaned_transformed.csv     :   6159.8 KB
✓ meteorites_cleaned_transformed.json    :  16635.0 KB
✓ sentry_risks_transformed.json          :     24.9 KB
